# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.


In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the Croissant dataset
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata
print(f"{metadata.name}: {metadata.description}")

## 2. Data Overview
Review available record sets, fields, and their IDs.

Below, we list all available record sets and their fields by their `@id`, following the Croissant conventions.

In [ ]:
# List all record sets (@id) and their fields (@id)
all_record_sets = dataset.metadata.record_sets
record_set_ids = []
for rs in all_record_sets:
    print(f"Record Set: {rs['@id']}")
    record_set_ids.append(rs['@id'])
    fields = rs.get('fields', [])
    print("  Fields and Columns (by @id):")
    for field in fields:
        print(f"    - Field @id: {field['@id']}, Name: {field.get('name', '<no name>')}")
        if 'columns' in field:
            for col in field['columns']:
                print(f"        - Column @id: {col['@id']}, Name: {col.get('name', '<no name>')}")
    print()
if not record_set_ids:
    print('No record sets found in the schema or Croissant version not exposing them via record_sets. Consider checking the dataset structure in detail.')

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis.

Use the first record set by its `@id` (you can modify as needed), and display its columns.

In [ ]:
# Use the first record set for demonstration
if record_set_ids:
    example_record_set_id = record_set_ids[0]
else:
    # Fallback: try typical key names if record_sets are missing in the schema.
    print("No record_set IDs found. Please check the Croissant metadata.")
    example_record_set_id = None
# Extract records for the selected record set
if example_record_set_id:
    records = list(dataset.records(record_set=example_record_set_id))
    df = pd.DataFrame(records)
    print(f"Loaded DataFrame with {len(df)} rows and columns:")
    print(df.columns.tolist())
    display(df.head())
else:
    print('No record set could be loaded.')

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data.

This example selects a numeric field (e.g., 'cr:peptide_count_total' or similar), filters, normalizes, and groups as appropriate.

> **Note:** Replace `<numeric_field_id>` and `<group_field_id>` below with actual field `@id`s of numeric and category fields found in your dataset.

In [ ]:
# Example field IDs (replace according to your overview output):
numeric_field_id = None
group_field_id = None
for col in df.columns:
    # Heuristic: select field names/ids that contain 'peptide', 'abundance', 'mw', etc.
    if any(key in col.lower() for key in ['peptide', 'abundance', 'coverage', 'mw']):
        numeric_field_id = col
        break
for col in df.columns:
    # Heuristic: select a group field, e.g., 'cr:condition' or 'cr:protein_group' or similar
    if any(key in col.lower() for key in ['sample', 'condition', 'group']):
        group_field_id = col
        break
if numeric_field_id is not None:
    threshold = df[numeric_field_id].mean() if pd.api.types.is_numeric_dtype(df[numeric_field_id]) else 10
    # Try to convert if the column is not already numeric (Croissant might load as string)
    if not pd.api.types.is_numeric_dtype(df[numeric_field_id]):
        df[numeric_field_id] = pd.to_numeric(df[numeric_field_id], errors='coerce')
    filtered_df = df[df[numeric_field_id] > threshold]
    print(f"Filtered records with {numeric_field_id} > {threshold}:")
    display(filtered_df.head())
    filtered_df[f"{numeric_field_id}_normalized"] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
    print(f"Normalized {numeric_field_id} for filtered records:")
    display(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())
    if group_field_id and group_field_id in filtered_df.columns:
        grouped_df = filtered_df.groupby(group_field_id).mean(numeric_only=True)
        print(f"Grouped data by {group_field_id}:")
        display(grouped_df.head())
else:
    print("No numeric field found for EDA. Review previous cells to select the correct field `@id`.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

Below, for demonstration we plot the distribution of the selected numeric field and, if available, compare groupings if a categorical/grouping field is present.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if numeric_field_id is not None:
    plt.figure(figsize=(8, 4))
    sns.histplot(df[numeric_field_id], kde=True, bins=30)
    plt.title(f"Distribution of {numeric_field_id}")
    plt.xlabel(numeric_field_id)
    plt.ylabel("Count")
    plt.show()
    if group_field_id and group_field_id in df.columns:
        plt.figure(figsize=(10,6))
        sns.boxplot(x=group_field_id, y=numeric_field_id, data=df)
        plt.title(f"{numeric_field_id} by {group_field_id}")
        plt.xticks(rotation=45)
        plt.tight_layout()
        plt.show()
else:
    print('No numeric field selected for visualization.')

## 6. Conclusion
Summarize key findings and observations from the dataset exploration.

- The dataset was loaded using a Croissant schema and the `mlcroissant` library.
- We explored its structure, identifying available record sets and fields by their unique `@id`.
- Using data extracted via record set and field IDs, we demonstrated EDA and basic visualizations.
- For further analysis, explore additional record sets and advanced ML workflows using the same referencing approach by `@id`.